# Teaching Computers to See : classification Cats vs Dogs avec un CNN

Objectif : construire, entraîner et évaluer un Convolutional Neural Network capable de distinguer des images de chats et de chiens, puis essayer d'améliorer ses performances.

## Step 1 : Introduction aux CNN

Les Convolutional Neural Networks (CNN) ont été pensés pour les images. Au lieu de regarder tous les pixels d'un coup, ils analysent l'image par petites zones, et ils construisent petit à petit une compréhension de ce qu'elle contient. Un CNN repose sur quatre types de couches.

**Les couches de convolution** sont le cœur du réseau. Un petit filtre, glisse sur toute l'image et calcule à chaque position une somme pondérée des pixels qu'il recouvre. Chaque filtre apprend à repérer un motif précis, par exemple un bord vertical ou une texture, et produit une "feature map" qui indique où ce motif apparaît dans l'image. L'avantage, c'est que le même filtre est réutilisé partout : un bord se détecte de la même façon en haut ou en bas de l'image. C'est ce qui permet aux CNN d'avoir beaucoup moins de paramètres qu'un réseau classique.

**Les couches de pooling** servent à réduire la taille des feature maps. Le MaxPooling, que l'on utilise, découpe l'image en carrés de 2 × 2 et ne garde que la valeur la plus forte de chaque carré. On divise ainsi la taille par deux en gardant l'information importante. Ça allège les calculs, et ça rend le modèle moins sensible aux petits décalages : peu importe que l'oreille du chat soit un pixel plus à gauche ou plus à droite.

**Les fonctions d'activation** sont appliquées après chaque convolution. On utilise la ReLU, qui garde les valeurs positives et remplace les négatives par 0. Sans elle, empiler des couches reviendrait à faire une seule grosse opération linéaire, et le réseau ne pourrait pas apprendre des formes complexes. La ReLU a aussi l'avantage de bien transmettre le gradient pendant l'apprentissage, contrairement à la sigmoid qui a tendance à l'écraser dans les réseaux profonds (le problème du vanishing gradient).

**Les couches fully connected** arrivent à la fin. Une fois que les convolutions ont extrait les caractéristiques de l'image, on les aplatit en un seul vecteur (avec une couche Flatten), puis des couches denses classiques les combinent pour prendre la décision finale : chat ou chien.

Ce qui rend les CNN puissants, c'est cet empilement. Les premières couches détectent des choses très simples, comme des bords ou des couleurs. Les suivantes combinent ces bords pour former des textures et des formes, et les dernières reconnaissent des éléments beaucoup plus abstraits, comme des yeux, des oreilles ou un museau. Le réseau apprend tout ça seul, uniquement à partir des exemples qu'on lui donne.

## Imports

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

## Step 2 : Exploration du dataset

On définit les chemins vers les dossiers `train` et `validation`, puis on compte le nombre d'images de chaque classe (cats, dogs) dans chacun. On affiche aussi la taille de 20 images de chats pour vérifier si elles ont toutes les mêmes dimensions.

In [ ]:
dataset_dir = "data/cats_and_dogs_filtered"
train_dir = os.path.join(dataset_dir, "train")
validation_dir = os.path.join(dataset_dir, "validation")

for split_name, split_dir in [("train", train_dir), ("validation", validation_dir)]:
    for class_name in ["cats", "dogs"]:
        class_dir = os.path.join(split_dir, class_name)
        print(f"{split_name}/{class_name}: {len(os.listdir(class_dir))} images")

cats_dir = os.path.join(train_dir, "cats")
image_sizes = [Image.open(os.path.join(cats_dir, filename)).size for filename in os.listdir(cats_dir)[:20]]
print(image_sizes)

### Analyse

Le dossier `train` contient 2000 images (1000 cats, 1000 dogs) et le dossier `validation` 1000 images (500 cats, 500 dogs). Les classes sont parfaitement équilibrées : l'accuracy sera donc une métrique fiable, et aucune pondération des classes n'est nécessaire.

Les images sont en couleur (3 canaux RGB), mais leurs dimensions varient d'une image à l'autre. Un CNN demande une entrée de taille fixe, donc toutes les images seront redimensionnées en 150×150 au chargement.

### Visualisation de quelques images

On affiche 4 images tirées au hasard pour chaque classe (cats sur la première ligne, dogs sur la seconde), avec leurs dimensions en titre, pour se faire une idée concrète des données.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row_index, class_name in enumerate(["cats", "dogs"]):
    class_dir = os.path.join(train_dir, class_name)
    for ax, filename in zip(axes[row_index], random.sample(os.listdir(class_dir), 4)):
        image = Image.open(os.path.join(class_dir, filename))
        ax.imshow(image)
        ax.set_title(f"{class_name} {image.size}")
        ax.axis("off")
plt.tight_layout()
plt.show()

On observe une grande variété dans les images : cadrages différents, arrière-plans plus ou moins encombrés, animaux partiellement visibles ou accompagnés d'humains, éclairages variables. Les dimensions changent aussi d'une image à l'autre, ce qui confirme la nécessité du redimensionnement. La tâche n'est donc pas triviale : le modèle devra apprendre des caractéristiques propres à l'animal, et non à l'arrière-plan.

### Split 80/20 et chargement des données

On charge les images avec `image_dataset_from_directory`, qui associe automatiquement chaque image à sa classe d'après le nom de son dossier.

- **Redimensionnement en 150×150** : assez grand pour garder les détails utiles (oreilles, museau, pelage), assez petit pour que l'entraînement reste raisonnable.
- **Split 80/20 du dossier `train`** : 80 % pour l'entraînement (1600 images) et 20 % pour le test (400 images). Le test set n'est utilisé qu'à l'évaluation finale, ce qui donne une mesure honnête des

In [ ]:
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 100

train_dataset, test_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=42,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

print(train_dataset.class_names)
for images, labels in train_dataset.take(1):
    print(images.shape, labels.shape)
    image_counts = {}

train_dataset = train_dataset.cache().prefetch(tf.data.AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(tf.data.AUTOTUNE)

for split_name, split_dir in [("train", train_dir), ("validation", validation_dir)]:
    for class_name in ["cats", "dogs"]:
        image_counts[f"{split_name}/{class_name}"] = len(os.listdir(os.path.join(split_dir, class_name)))

plt.figure(figsize=(8, 4))
plt.bar(image_counts.keys(), image_counts.values(), color=["tab:blue", "tab:orange", "tab:blue", "tab:orange"])
plt.title("Distribution des classes")
plt.ylabel("Nombre d'images")
plt.show()

Keras affiche "Using 400 files for validation" pour le deuxième sous-ensemble du split : dans notre cas, ces 400 images constituent bien le **test set**, la validation étant le dossier séparé de 1000 images.

Les classes sont détectées dans l'ordre alphabétique : `cats` = 0 et `dogs` = 1. Un batch a la forme `(100, 150, 150, 3)` : 100 images de 150×150 pixels sur 3 canaux RGB. Les labels ont la forme `(100, 1)`, une valeur binaire par image. L'input shape du modèle sera donc `(150, 150, 3)`.

## Step 3 : Construction du CNN

Pour notre modèle, on a suivi la structure demandée (4 blocs de convolution, du pooling, de la ReLU et une couche dense en sortie), mais en essayant de faire des choix adaptés à notre situation : un petit dataset de 1600 images d'entraînement.

On commence par une couche `Rescaling` qui ramène les pixels entre 0 et 1. Les réseaux apprennent mieux avec des valeurs petites, et comme cette étape fait partie du modèle, elle sera aussi appliquée automatiquement quand on fera des prédictions.

Chaque bloc contient une convolution 3×3 suivie d'un MaxPooling 2×2. On a choisi de doubler le nombre de filtres à chaque bloc (16, 32, 64, puis 128) : au début, le réseau cherche des choses simples comme des bords ou des couleurs, et il n'a pas besoin de beaucoup de filtres pour ça. Plus on avance, plus les motifs deviennent complexes (une oreille, un museau), et plus il faut de filtres pour les représenter. Commencer à 16 plutôt qu'à 32 permet aussi de garder un modèle léger.

On utilise `padding="same"` pour que les convolutions ne rognent pas les bords de l'image. Sans ça, chaque convolution enlève un pixel de chaque côté, et un chat placé au bord de la photo pourrait perdre des informations. Ici, c'est uniquement le pooling qui réduit la taille, ce qui rend l'évolution des dimensions plus simple à suivre.

La ReLU est utilisée partout comme activation : elle est très simple à calculer et évite le problème du vanishing gradient qu'on aurait avec une sigmoid dans les couches cachées.

Pour la partie fully connected, on a mis une seule couche Dense de 128 neurones. On aurait pu en mettre 512, mais vu le peu d'images qu'on a, un modèle trop gros risque surtout d'apprendre les images par cœur. Enfin, la sortie est un seul neurone avec une sigmoid : il renvoie une probabilité entre 0 et 1, et on considère que l'image est un chien si elle dépasse 0.5.

In [ ]:
tf.keras.utils.set_random_seed(42)

baseline_model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Rescaling(1.0 / 255),

    layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

In [ ]:
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
baseline_initial_weights = baseline_model.get_weights()

baseline_model.summary()

### Analyse du modèle

Le résumé se lit de haut en bas, dans l'ordre où l'image traverse le réseau. Dans la colonne Output Shape, le `None` correspond à la taille du batch, qui n'est pas fixée à l'avance : les trois autres valeurs sont la hauteur, la largeur et le nombre de canaux.

| Couche | Sortie | Paramètres | Rôle |
|---|---|---|---|
| Rescaling | 150 × 150 × 3 | 0 | Ramène les pixels entre 0 et 1. Simple calcul fixe, rien à apprendre. |
| Conv2D (16) | 150 × 150 × 16 | 448 | 16 filtres détectent des motifs simples (bords, couleurs). La taille est conservée grâce au `padding="same"`. |
| MaxPooling | 75 × 75 × 16 | 0 | Divise la taille par 2 en gardant les activations les plus fortes. |
| Conv2D (32) | 75 × 75 × 32 | 4 640 | 32 filtres combinent les motifs précédents en textures et formes simples. |
| MaxPooling | 37 × 37 × 32 | 0 | 75 / 2 arrondi à l'inférieur. |
| Conv2D (64) | 37 × 37 × 64 | 18 496 | Des motifs plus complexes, comme des parties d'animal. |
| MaxPooling | 18 × 18 × 64 | 0 | Nouvelle division par 2. |
| Conv2D (128) | 18 × 18 × 128 | 73 856 | Les motifs les plus abstraits (oreilles, museau, forme de la tête). |
| MaxPooling | 9 × 9 × 128 | 0 | Il reste 128 feature maps de 9 × 9. |
| Flatten | 10 368 | 0 | Déplie le bloc 9 × 9 × 128 en un vecteur, pour pouvoir le passer à une couche dense. |
| Dense (128) | 128 | 1 327 232 | Combine toutes les caractéristiques extraites pour prendre la décision. C'est la couche la plus lourde. |
| Dense (1) | 1 | 129 | Neurone de sortie avec sigmoid : la probabilité que l'image soit un chien. |

Grâce au `padding="same"`, les convolutions gardent la même taille d'image et seul le pooling la divise par 2 (de 150 × 150 à 75, puis 37, 18 et enfin 9 × 9). En parallèle, le nombre de canaux augmente de 3 à 128 : le réseau perd en résolution spatiale mais gagne en richesse d'information, chaque position décrivant des caractéristiques de plus en plus abstraites.

Le modèle compte 1 424 801 paramètres au total, tous entraînables, pour environ 5,4 Mo.

La répartition des paramètres est très déséquilibrée. Les quatre couches de convolution en totalisent moins de 100 000, alors que la couche Dense après le Flatten en contient à elle seule 1 327 232, soit environ 93 %. Les convolutions sont économes parce qu'elles réutilisent les mêmes petits filtres sur toute l'image, alors que la couche dense connecte chacune des 10 368 valeurs à chacun des 128 neurones. C'est donc surtout cette couche qui pourrait mémoriser les images d'entraînement au lieu d'apprendre des caractéristiques générales. On s'attend à voir un peu d'overfitting à l'entraînement.

## Step 4 : Entraînement du modèle

On entraîne le modèle :
- sur 15 epochs, la limite haute demandée, pour bien voir l'évolution des courbes et repérer un éventuel overfitting, qui n'apparaît souvent qu'après plusieurs epochs
- avec un batch size de 100, imposé par le sujet

À chaque epoch, le modèle est évalué sur le dossier de validation, pour comparer ce qu'il apprend sur les images d'entraînement avec ce qu'il réussit sur des images qu'il ne voit pas pendant l'apprentissage.

Juste après le chargement des données (Step 2), on a ajouté `cache()` et `prefetch()` sur les datasets : les images sont gardées en mémoire après la première epoch, et le batch suivant est préparé pendant que le modèle travaille sur le batch actuel. Comme on est sur CPU, ça fait gagner pas mal de temps.

### Calcul du nombre de paramètres

Pour vérifier qu'on comprend bien ce que fait le modèle, on a recalculé le nombre de paramètres à la main.

Pour une convolution, chaque filtre a un poids par pixel du kernel et par canal d'entrée, plus un biais : (3 × 3 × canaux_entrée + 1) × nombre_de_filtres. Pour une couche dense, c'est (nombre_d'entrées + 1) × nombre_de_neurones. Les couches Rescaling, MaxPooling et Flatten n'ont rien à apprendre, donc zéro paramètre.

Grâce au `padding="same"`, seule la taille change au pooling : 150 → 75 → 37 → 18 → 9. À la sortie du dernier bloc, on a donc des feature maps de 9 × 9 avec 128 canaux.

| Couche | Calcul | Paramètres |
|---|---|---|
| Conv2D 1 | (3×3×3 + 1) × 16 | 448 |
| Conv2D 2 | (3×3×16 + 1) × 32 | 4 640 |
| Conv2D 3 | (3×3×32 + 1) × 64 | 18 496 |
| Conv2D 4 | (3×3×64 + 1) × 128 | 73 856 |
| Flatten | 9 × 9 × 128 = 10 368 valeurs | 0 |
| Dense 128 | (10 368 + 1) × 128 | 1 327 232 |
| Dense 1 | (128 + 1) × 1 | 129 |
| **Total** | | **1 424 801** |

On retrouve bien le même total que `baseline_model.summary()`. Ce qui nous a surpris, c'est qu'environ 93 % des paramètres se trouvent dans la seule couche Dense après le Flatten, alors que les quatre convolutions n'en ont même pas 100 000 à elles toutes. C'est donc surtout cette couche qui risque de faire de l'overfitting, et c'est un point qu'on gardera en tête pour la suite.

In [ ]:
EPOCHS = 15

def reset_training(model, initial_weights):
    model.set_weights(initial_weights)
    model.compile_from_config(model.get_compile_config())


tf.keras.utils.set_random_seed(42)
reset_training(baseline_model, baseline_initial_weights)

baseline_history = baseline_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    shuffle=False,
)

### Lecture de l'entraînement

Pour chaque epoch, Keras affiche une ligne avec quatre valeurs :
- `accuracy` et `loss` : les performances sur les images d'entraînement, calculées pendant l'epoch ;
- `val_accuracy` et `val_loss` : les performances sur le dossier de validation, mesurées à la fin de l'epoch, sur des images que le modèle n'utilise pas pour apprendre.

Le `16/16` correspond aux 16 batchs de 100 images qui composent nos 1600 images d'entraînement. Chaque epoch prend environ 2 à 4 secondes : le modèle est assez léger pour s'entraîner rapidement sur CPU.

En lisant les chiffres, on voit déjà deux phases. Pendant les six ou sept premières epochs, l'entraînement et la validation progressent ensemble : on passe d'environ 50 %, c'est-à-dire le hasard, à près de 68 % des deux côtés. Ensuite, l'accuracy d'entraînement continue de monter jusqu'à 84,3 %, alors que celle de validation stagne autour de 72-73 % à partir de l'epoch 9 et termine à 72,5 %. La loss d'entraînement passe de 0,70 à 0,36, mais celle de validation se stabilise autour de 0,54 à partir de l'epoch 9 et remonte même légèrement sur les dernières epochs.

Le modèle continue donc à s'améliorer sur les images qu'il connaît, sans progresser sur les nouvelles. Les courbes ci-dessous permettent de voir plus précisément à quel moment ça se produit.

### Courbes d'apprentissage

On trace l'accuracy et la loss au fil des epochs, pour l'entraînement et la validation. On les met dans une fonction pour pouvoir la réutiliser à la Step 6 et comparer les versions du modèle.

In [ ]:
def plot_history(history, title):
    training_metrics = history.history
    epoch_numbers = range(1, len(training_metrics["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epoch_numbers, training_metrics["accuracy"], label="train")
    axes[0].plot(epoch_numbers, training_metrics["val_accuracy"], label="validation")
    axes[0].set_title(f"{title} : accuracy")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(epoch_numbers, training_metrics["loss"], label="train")
    axes[1].plot(epoch_numbers, training_metrics["val_loss"], label="validation")
    axes[1].set_title(f"{title} : loss")
    axes[1].set_xlabel("epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_history(baseline_history, "Baseline")

### Analyse : overfitting ou underfitting ?

Les courbes confirment ce qu'on voyait dans les chiffres : notre modèle fait de l'**overfitting**. À la fin, il y a 11,8 points d'écart entre l'entraînement et la validation (84,3 % contre 72,5 %). Ce n'est pas de l'underfitting, puisque le modèle obtient de bons résultats sur les images d'entraînement : il est capable d'apprendre, mais une partie de ce qu'il apprend ne sert qu'à reconnaître ces images-là.

Les courbes permettent surtout de voir à quel moment ça bascule. Au début, la validation est même au-dessus de l'entraînement (epochs 1 à 6). C'est un effet de la façon dont Keras calcule les scores : l'accuracy d'entraînement est une moyenne prise pendant l'epoch, alors que le modèle s'améliore encore, tandis que la validation est mesurée à la fin, avec le modèle déjà mis à jour.

Les deux courbes d'accuracy se croisent à l'epoch 7, et l'écart grandit ensuite presque sans interruption : 2,4 points à l'epoch 8, 5,2 à l'epoch 11, puis 11,8 à la fin. Sur la loss, c'est encore plus net : les courbes se croisent aussi à l'epoch 7, puis prennent deux directions opposées. Celle d'entraînement continue de descendre presque en ligne droite jusqu'à 0,36, alors que celle de validation reste à plat entre 0,54 et 0,56 à partir de l'epoch 9. Elle atteint son minimum à l'epoch 11 (0,539), puis remonte légèrement jusqu'à 0,560. Ce plateau montre que le modèle ne progresse plus vraiment sur les nouvelles images : il fluctue un peu d'une epoch à l'autre, sans tendance claire.

On peut l'expliquer avec ce qu'on avait vu à la Step 3 : seulement 1600 images d'entraînement pour 1,4 million de paramètres, dont 93 % dans une seule couche dense, et aucune régularisation. En plus, le modèle revoit exactement les mêmes images à chaque epoch, ce qui lui permet de les mémoriser petit à petit.

L'overfitting reste quand même modéré, sans doute grâce à nos choix pour garder le modèle léger (Dense de 128, learning rate réduit). Mais il limite clairement les performances : à partir de l'epoch 9, continuer l'entraînement n'apporte plus rien en validation (72,5 % à l'epoch 9 comme à l'epoch 15, avec un maximum à 73,7 %). C'est ce qu'on va essayer de corriger à la Step 6, avec la data augmentation puis le dropout et la batch normalization.

## Step 5 : Évaluation sur le test set

Jusqu'ici, on a seulement regardé la validation. On évalue maintenant le modèle sur les 400 images du test set, qu'il n'a jamais vues, ni pendant l'entraînement ni pour ajuster quoi que ce soit. C'est la mesure la plus honnête de ses performances.

Pour chaque image, le modèle renvoie une probabilité entre 0 et 1 : au-dessus de 0.5, on considère que c'est un chien, sinon un chat. On récupère les prédictions et les vrais labels dans la même boucle, pour être sûrs qu'ils restent bien alignés.

On regarde ensuite :
- la **matrice de confusion**, pour voir où le modèle se trompe
- l'**accuracy** : la proportion d'images bien classées
- la **precision** : parmi les images que le modèle classe "dog", combien sont vraiment des chiens
- le **recall** : parmi tous les vrais chiens, combien le modèle en retrouve
- le **F1-score** : un compromis entre precision et recall

In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)


def evaluate_model(model, dataset, title):
    true_labels, predicted_probabilities = [], []
    for images, labels in dataset:
        true_labels.append(labels.numpy().ravel())
        predicted_probabilities.append(model.predict(images, verbose=0).ravel())
    true_labels = np.concatenate(true_labels).astype(int)
    predicted_labels = (np.concatenate(predicted_probabilities) >= 0.5).astype(int)

    confusion = confusion_matrix(true_labels, predicted_labels)
    ConfusionMatrixDisplay(confusion, display_labels=["cats", "dogs"]).plot(cmap="Blues")
    plt.title(f"{title} : confusion matrix")
    plt.show()

    results = {
        "model": title,
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "precision": precision_score(true_labels, predicted_labels),
        "recall": recall_score(true_labels, predicted_labels),
        "f1": f1_score(true_labels, predicted_labels),
    }
    for metric_name in ["accuracy", "precision", "recall", "f1"]:
        print(f"{metric_name}: {results[metric_name]:.4f}")
    return results


baseline_results = evaluate_model(baseline_model, test_dataset, "Baseline")

### Analyse des résultats

Sur les 400 images du test set, le modèle obtient une accuracy de **77,2 %**. C'est un peu au-dessus de ce qu'on observait en validation (72,5 %), mais loin des 84,3 % obtenus sur les images d'entraînement : le modèle généralise nettement moins bien qu'il n'apprend.

On remarque que le test set n'est pas parfaitement équilibré (182 chats, 218 chiens), parce que le split 80/20 est aléatoire. L'écart reste faible, donc l'accuracy reste une métrique fiable.

La matrice de confusion montre que le modèle ne se trompe pas autant dans les deux sens : 55 chats sont classés comme des chiens, contre 36 chiens classés comme des chats. Il a donc une légère tendance à répondre "chien" quand il hésite. Le taux de réussite est meilleur pour les chiens (182 sur 218, soit 83,5 %) que pour les chats (127 sur 182, soit 70 %).

Les métriques le confirment :
- **Precision de 76,8 %** : quand le modèle dit "chien", il a raison un peu plus de trois fois sur quatre.
- **Recall de 83,5 %** : il retrouve plus de 8 chiens sur 10.
- **F1-score de 80,0 %** : le compromis entre les deux. Le recall est plus élevé que la precision, ce qui traduit la tendance à répondre "chien".

Au final, le modèle fait nettement mieux que le hasard, mais un peu moins d'une image sur quatre est mal classée. Il a appris des caractéristiques trop liées aux images d'entraînement, comme on l'a vu avec l'overfitting à la Step 4. C'est ce problème que la Step 6 va essayer de corriger.

## Step 6 : Amélioration du modèle

Le baseline fait de l'overfitting : 84 % sur l'entraînement contre 72,5 % en validation, soit 11,8 points d'écart. On teste maintenant deux techniques pour réduire cet écart, **séparément**, pour pouvoir mesurer l'effet de chacune.

Pour que la comparaison soit honnête, on ne change qu'une seule chose à la fois : même architecture, même learning rate (0.0005), mêmes 15 epochs, même batch size et même split. On fixe aussi la seed aléatoire avant chaque entraînement, pour que les différences viennent bien de la technique testée et pas du hasard de l'initialisation.

Plutôt que de recopier l'architecture à chaque fois, on écrit une fonction `build_model` qui construit le modèle de base, avec deux options : ajouter la data augmentation, ou ajouter dropout et batch normalization.

In [ ]:
def build_data_augmentation():
    return models.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.15),
    ], name="data_augmentation")


data_augmentation = build_data_augmentation()


def build_model(augment=False, regularize=False):
    cnn = models.Sequential([layers.Input(shape=(150, 150, 3))])

    if augment:
        cnn.add(build_data_augmentation())

    cnn.add(layers.Rescaling(1.0 / 255))

    for filters in [16, 32, 64, 128]:
        if regularize:
            cnn.add(layers.Conv2D(filters, (3, 3), padding="same", use_bias=False))
            cnn.add(layers.BatchNormalization(momentum=0.9))
            cnn.add(layers.Activation("relu"))
        else:
            cnn.add(layers.Conv2D(filters, (3, 3), padding="same", activation="relu"))
        cnn.add(layers.MaxPooling2D((2, 2)))

    cnn.add(layers.Flatten())
    if regularize:
        cnn.add(layers.Dropout(0.5))
    cnn.add(layers.Dense(128, activation="relu"))
    cnn.add(layers.Dense(1, activation="sigmoid"))

    cnn.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return cnn

### 6.A : Data augmentation

L'idée est de montrer au modèle, à chaque epoch, des versions légèrement modifiées des images d'entraînement. Il ne voit jamais deux fois exactement la même image, ce qui l'empêche de les apprendre par cœur et l'oblige à se concentrer sur ce qui fait vraiment un chat ou un chien.

On a choisi des transformations réalistes pour des photos d'animaux :
- **Flip horizontal** : un chat tourné vers la gauche reste un chat. On n'utilise pas de flip vertical, parce qu'on ne voit jamais d'animal la tête en bas sur une photo.
- **Rotation jusqu'à ±10 %** (environ ±36°) : les photos ne sont pas toujours prises bien droites.
- **Zoom jusqu'à ±15 %** : l'animal peut être plus ou moins près de l'objectif.

Ces transformations sont des couches Keras placées au début du modèle. Elles ne sont actives que pendant l'entraînement : lors de l'évaluation, sur la validation comme sur le test set, les images passent sans être modifiées.

Le sujet indique d'appliquer l'augmentation à l'entraînement et à la validation. On a choisi de ne l'appliquer qu'à l'entraînement, ce qui est la pratique standard : la validation sert à mesurer comment le modèle se comporte sur de vraies images, et la déformer fausserait cette mesure. Dans tous les cas, le test set n'est jamais augmenté.

Pour voir concrètement ce que reçoit le modèle, on affiche une même image avec plusieurs augmentations aléatoires.

In [ ]:
for images, labels in train_dataset.take(1):
    sample_image = images[0]

plt.figure(figsize=(12, 6))
plt.subplot(2, 4, 1)
plt.imshow(sample_image.numpy().astype("uint8"))
plt.title("original")
plt.axis("off")

for i in range(1, 8):
    augmented_images = data_augmentation(tf.expand_dims(sample_image, 0), training=True)
    plt.subplot(2, 4, i + 1)
    plt.imshow(augmented_images[0].numpy().astype("uint8"))
    plt.title(f"augmentation {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
tf.keras.utils.set_random_seed(42)

augmented_model = build_model(augment=True)

augmented_history = augmented_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    shuffle=False,
)

plot_history(augmented_history, "Data augmentation")

#### Analyse des courbes

L'effet sur l'overfitting est très net. Avec le baseline, l'écart entre l'entraînement et la validation atteignait 11,8 points à la fin. Avec la data augmentation, il tombe à 4,2 points (73,1 % contre 68,9 %). Les deux courbes restent proches pendant tout l'entraînement : jusqu'à l'epoch 6, la validation est même toujours au-dessus de l'entraînement, et elle repasse encore devant aux epochs 8 et 10.

La loss le confirme. Avec le baseline, la loss d'entraînement descendait jusqu'à 0,36 pendant que celle de validation restait bloquée autour de 0,55. Ici, les deux restent proches jusqu'au bout : 0,54 pour l'entraînement et 0,57 pour la validation à la dernière epoch. Le modèle n'apprend plus les images par cœur : comme il voit une version différente de chaque image à chaque epoch, il est obligé de retenir des caractéristiques plus générales.

En revanche, à 15 epochs, la data augmentation fait un peu moins bien que le baseline en validation : 68,9 % contre 72,5 %, avec une loss de validation minimale légèrement plus haute (0,549 contre 0,539). L'accuracy d'entraînement est aussi plus basse (73,1 % contre 84,3 %). C'est normal : la tâche est devenue plus difficile, puisque le modèle doit reconnaître des animaux retournés, tournés ou zoomés, et il apprend donc plus lentement.

L'apprentissage est effectivement plus lent : l'accuracy d'entraînement progressait encore sur les dernières epochs (de 70 % à l'epoch 12 à 73 % à la fin). La validation, elle, stagne autour de 69-71 % depuis l'epoch 8, et sa loss remonte légèrement après son minimum de l'epoch 13. On ne peut donc pas affirmer que plus d'epochs suffiraient à dépasser le baseline. Ce qui est sûr, c'est qu'en 15 epochs, la limite imposée par le sujet, cette technique réduit l'overfitting sans améliorer le score de validation.

In [ ]:
augmented_results = evaluate_model(augmented_model, test_dataset, "Data augmentation")

#### Analyse sur le test set

Sur le test set, la data augmentation obtient une accuracy de **73,5 %**, en dessous du baseline (77,2 %). La différence représente 15 images sur 400. Le F1-score baisse moins, de 80,0 % à **78,2 %**.

La matrice de confusion explique ce résultat. Le baseline avait déjà une légère tendance à répondre "chien" (55 chats classés comme des chiens). Avec l'augmentation, cette tendance est beaucoup plus marquée : le modèle reconnaît très bien les chiens (190 sur 218, soit 87,2 %), mais il classe 78 chats sur 182 comme des chiens, et n'en reconnaît donc que 57 %.

Les métriques reflètent ce déséquilibre :
- **Recall de 87,2 %** (contre 83,5 % pour le baseline) : le modèle retrouve près de 9 vrais chiens sur 10.
- **Precision de 70,9 %** (contre 76,8 %) : en contrepartie, quand il dit "chien", il se trompe près de trois fois sur dix.
- **F1-score de 78,2 %** : il recule un peu, parce que la perte en precision est plus importante que le gain en recall.

Ce biais est cohérent avec ce qu'on a vu sur les courbes : le modèle apprend plus lentement, et à l'epoch 15 sa frontière de décision n'est pas encore bien placée entre les deux classes.

Il faut rester prudent avant de conclure sur ces écarts. Lors de nos premiers essais, avant de fixer toutes les seeds et de rendre les calculs déterministes, les résultats de la data augmentation variaient beaucoup d'un entraînement à l'autre : le même modèle avait par exemple atteint 80,8 % sur le test set, avec des erreurs bien réparties entre les deux classes. Comme les transformations sont tirées au hasard, un autre tirage peut donner un résultat différent de quelques points.

### 6.B : Dropout et Batch Normalization

On repart du modèle de base, sans augmentation, et on ajoute deux autres techniques.

**Batch Normalization** : après chaque convolution, elle normalise les sorties pour qu'elles gardent une moyenne et une variance stables pendant l'entraînement. Sans elle, chaque couche doit sans arrêt s'adapter aux changements des couches précédentes. Avec elle, l'apprentissage est plus stable et converge en général plus vite.

On l'a placée entre la convolution et la ReLU (Conv2D → BatchNormalization → ReLU), comme dans l'article original. La convolution n'a plus de biais, parce que la batch normalization ajoute déjà son propre décalage.

Un point important : la batch normalization ne fonctionne pas de la même façon pendant l'entraînement et pendant l'évaluation. À l'entraînement, elle utilise les statistiques du batch en cours. À l'évaluation, elle utilise une moyenne glissante de ces statistiques, calculée au fil de l'entraînement. Par défaut, cette moyenne évolue très lentement (momentum de 0.99). Comme on n'a que 16 batchs par epoch, on l'a passée à 0.9 pour qu'elle suive plus vite l'évolution du modèle. Lors d'un premier essai avec la valeur par défaut, le modèle atteignait 98,7 % sur l'entraînement mais restait bloqué à 50 % en validation : les statistiques utilisées en évaluation ne correspondaient pas du tout à celles vues pendant l'entraînement.

**Dropout** : à chaque étape d'entraînement, il désactive au hasard une partie des neurones. Le réseau ne peut plus compter sur quelques neurones précis et doit répartir ce qu'il apprend. On l'a placé juste avant la couche Dense de 128, avec un taux de 0.5 : c'est cette couche qui contient 93 % des paramètres, donc c'est là que le risque de mémorisation est le plus fort. Comme l'augmentation, le dropout n'est actif que pendant l'entraînement.

In [ ]:
tf.keras.utils.set_random_seed(42)

regularized_model = build_model(regularize=True)
regularized_model.summary()

regularized_history = regularized_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    shuffle=False,
)

plot_history(regularized_history, "Dropout + BatchNorm")

#### Analyse des courbes

Le contraste avec nos CNN est frappant. Dès la première epoch, le modèle atteint 93,8 % en validation, alors que nos CNN démarraient autour de 50 %, c'est-à-dire au hasard. Il termine à **98 % en validation** après seulement 5 epochs, contre 69 à 76 % pour nos CNN après 15 epochs. La loss de validation passe de 0,27 à 0,08.

Le modèle n'a presque rien à apprendre : les convolutions de MobileNetV2 savent déjà extraire des caractéristiques très utiles (formes de tête, oreilles, museau, pelage), puisqu'elles ont été entraînées sur plus d'un million d'images, dont de nombreux animaux. La seule chose qu'on entraîne, c'est la dernière couche, qui apprend à combiner ces 1280 caractéristiques pour distinguer un chat d'un chien.

On remarque que la validation reste au-dessus de l'entraînement pendant toutes les epochs (98 % contre 97 % à la fin). Ce n'est pas un problème, et ça s'explique de deux façons. D'abord, l'accuracy d'entraînement est une moyenne calculée pendant l'epoch, alors que le modèle s'améliore encore : c'est surtout visible à la première epoch (72,8 % en entraînement contre 93,8 % en validation), où le modèle part de zéro. Ensuite, le dropout est actif pendant l'entraînement et désactive 20 % des caractéristiques, ce qui rend la tâche un peu plus difficile qu'en validation, où toutes les caractéristiques sont utilisées.

Il n'y a donc **aucun overfitting** : le modèle réussit au moins aussi bien sur les images qu'il n'a jamais vues que sur celles d'entraînement. Les courbes continuent même de progresser légèrement à la 5e epoch, mais le gain devient très faible (de 97,9 % à 98 % entre les epochs 4 et 5), donc entraîner plus longtemps n'apporterait presque rien.

Seul point négatif : chaque epoch prend environ 6 à 9 secondes, contre 2 à 4 pour notre CNN de base. Même si on n'entraîne qu'une seule couche, chaque image doit passer entièrement à travers MobileNetV2, qui est un réseau beaucoup plus profond. Sur 5 epochs, l'entraînement dure quand même à peu près aussi longtemps que celui de notre baseline sur 15 epochs.

In [ ]:
regularized_results = evaluate_model(regularized_model, test_dataset, "Dropout + BatchNorm")

#### Analyse sur le test set

Sur le test set, ce modèle obtient la meilleure accuracy des trois : **78,8 %**, contre 77,2 % pour le baseline et 73,5 % pour la data augmentation. L'écart avec le baseline ne représente que 6 images sur 400. Son F1-score (79,7 %) est équivalent à celui du baseline (80,0 %), et au-dessus de celui de la data augmentation (78,2 %).

Ce résultat est cohérent avec les courbes : ce modèle avait aussi la meilleure accuracy de validation à la fin de l'entraînement (75,5 %, contre 72,5 % pour le baseline). La réduction de l'overfitting se traduit donc par des performances légèrement meilleures sur des images jamais vues.

La matrice de confusion montre un comportement différent des deux autres modèles : celui-ci reconnaît mieux les chats (148 sur 182, soit 81 %) que les chiens (167 sur 218, soit 76,6 %). 51 chiens sont pris pour des chats, contre 34 chats pris pour des chiens. Il a donc une légère tendance à répondre "chat" quand il hésite, à l'inverse du baseline et surtout de la data augmentation, qui classait 78 chats comme des chiens.

- **Precision de 83,1 %** : quand le modèle dit "chien", il a raison plus de 8 fois sur 10. C'est la meilleure precision des trois modèles.
- **Recall de 76,6 %** : il retrouve un peu plus de trois chiens sur quatre, moins que les deux autres modèles.
- **F1-score de 79,7 %** : le gain en precision compense la perte en recall, et le F1-score reste au niveau du baseline.

Dropout et batch normalization réduisent donc nettement l'overfitting, avec un léger gain d'accuracy, et le dernier point de validation montre que le modèle progressait encore à l'epoch 15.

### Comparaison des trois modèles

In [ ]:
comparison = pd.DataFrame([baseline_results, augmented_results, regularized_results]).set_index("model")
comparison["écart train/val"] = [
    training_history.history["accuracy"][-1] - training_history.history["val_accuracy"][-1]
    for training_history in [baseline_history, augmented_history, regularized_history]
]
display(comparison.style.format("{:.1%}"))

comparison[["accuracy", "f1", "écart train/val"]].plot.bar(figsize=(10, 5), rot=0)
plt.title("Comparaison des modèles sur le test set")
plt.ylabel("score")
plt.ylim(0, 1)
plt.show()

#### Bilan de la Step 6

| Modèle | Accuracy | F1-score | Écart train/val |
|---|---|---|---|
| Baseline | 77,2 % | **80,0 %** | 11,8 pts |
| Data augmentation | 73,5 % | 78,2 % | 4,2 pts |
| Dropout + BatchNorm | **78,8 %** | 79,7 % | **3,5 pts** |

L'accuracy et le F1-score sont mesurés sur le test set. L'écart train/val est la différence entre l'accuracy d'entraînement et l'accuracy de validation à la dernière epoch : il mesure l'overfitting, et plus il est proche de 0, mieux le modèle généralise.

Les deux techniques remplissent leur objectif principal : elles réduisent fortement l'overfitting. L'écart entre entraînement et validation passe de 11,8 points pour le baseline à 4,2 points avec la data augmentation et à 3,5 points avec dropout et batch normalization. En revanche, en 15 epochs, cette réduction ne se traduit pas par un gain net sur le test set : dropout et batch normalization gagnent 1,5 point d'accuracy (6 images), la data augmentation en perd près de 4 (15 images), et le F1-score ne dépasse celui du baseline dans aucun des deux cas.

Elles y arrivent de deux façons différentes. La data augmentation agit sur les données : elle montre au modèle des images toujours différentes, qu'il ne peut pas mémoriser. Dropout et batch normalization agissent sur le réseau : le dropout l'empêche de trop compter sur certains neurones, et la batch normalization stabilise l'apprentissage. C'est cette combinaison qui réduit le plus l'overfitting et qui obtient la meilleure accuracy (78,8 %), avec des erreurs mieux réparties que la data augmentation.

Les trois modèles ne se trompent pas de la même façon. Le baseline et surtout la data augmentation ont un biais vers la classe "dog" (recall élevé, precision plus faible) : 55 et 78 chats classés comme des chiens. Dropout et batch normalization ont le biais inverse, plus léger (51 chiens classés comme des chats). Comme ces biais vont dans des sens opposés, ils ne viennent sans doute pas des techniques elles-mêmes : ils reflètent plutôt l'endroit où se trouve la frontière de décision de chaque modèle à l'epoch 15, qui peut encore bouger d'une epoch à l'autre.

Il faut aussi garder une limite en tête : chaque modèle n'a été entraîné qu'une seule fois, et le test set ne contient que 400 images, donc 1 point d'accuracy correspond à 4 images. Lors de nos premiers essais, avant de fixer toutes les seeds et de rendre les calculs déterministes, les résultats pouvaient varier de plusieurs points d'un lancement à l'autre. Les écarts de quelques points entre les trois modèles ne sont donc pas significatifs. La conclusion la plus solide est que les deux techniques réduisent nettement l'overfitting.

Pour aller plus loin, on pourrait entraîner plus longtemps, combiner les deux techniques, ou passer au transfer learning : partir d'un modèle déjà entraîné sur des millions d'images, plutôt que d'apprendre à partir de seulement 1600.

## Bonus : Transfer learning avec MobileNetV2

Jusqu'ici, notre CNN apprenait tout depuis zéro, avec seulement 1600 images. Le transfer learning consiste à repartir d'un modèle déjà entraîné sur un très grand dataset, et à réutiliser ce qu'il sait déjà.

On utilise MobileNetV2, recommandé par le sujet, entraîné sur ImageNet : 1,4 million d'images réparties en 1000 catégories, dont de nombreuses races de chats et de chiens. Ses couches de convolution savent déjà repérer des bords, des textures, des yeux, des oreilles… exactement ce que notre CNN essayait d'apprendre seul. C'est aussi un modèle léger, conçu à l'origine pour les téléphones, donc adapté à un entraînement sur CPU.

Notre démarche :
- On charge MobileNetV2 **sans sa dernière couche** (`include_top=False`), qui servait à choisir parmi les 1000 catégories d'ImageNet.
- On **gèle** toutes ses couches (`trainable = False`) : ses poids ne bougeront pas pendant l'entraînement. On garde ainsi intact ce qu'il a appris sur ImageNet.
- On ajoute par-dessus une petite tête de classification, la seule partie qu'on entraîne.

Quelques choix techniques :
- **Redimensionnement en 160 × 160** : les poids ImageNet de MobileNetV2 sont prévus pour certaines tailles précises (96, 128, 160, 192 ou 224). On garde nos datasets en 150 × 150 et on agrandit légèrement les images dans le modèle, ce qui évite de tout recharger.
- **Pixels entre -1 et 1** : c'est l'échelle utilisée pendant l'entraînement de MobileNetV2 sur ImageNet. On remplace donc notre `Rescaling(1/255)` par un `Rescaling(1/127.5, offset=-1)`.
- **`training=False`** : MobileNetV2 contient des couches de batch normalization. Comme on l'a vu à la Step 6, ces couches se comportent différemment à l'entraînement et à l'évaluation. On force ici le mode évaluation, pour qu'elles gardent les statistiques apprises sur ImageNet au lieu de les recalculer sur nos petits batchs.
- **`GlobalAveragePooling2D` au lieu de `Flatten`** : on fait la moyenne de chaque feature map au lieu de toutes les mettre à plat. C'est exactement le problème vu à la Step 3, où le Flatten créait une couche dense de 1,3 million de paramètres. Ici, la tête n'en a que 1281.
- **`Dropout(0.2)`** : une régularisation légère, puisque la tête est déjà très petite.
- **5 epochs** : le modèle part déjà d'une très bonne représentation des images, donc il converge beaucoup plus vite que notre CNN. 5 epochs suffisent, ce qui reste dans la fourchette du sujet.

In [ ]:
tf.keras.utils.set_random_seed(42)

mobilenet_base = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet",
)
mobilenet_base.trainable = False

inputs = layers.Input(shape=(150, 150, 3))
features = layers.Resizing(160, 160)(inputs)
features = layers.Rescaling(1.0 / 127.5, offset=-1)(features)
features = mobilenet_base(features, training=False)
features = layers.GlobalAveragePooling2D()(features)
features = layers.Dropout(0.2)(features)
outputs = layers.Dense(1, activation="sigmoid")(features)

mobilenet_model = models.Model(inputs, outputs)

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
mobilenet_initial_weights = mobilenet_model.get_weights()

mobilenet_model.summary()

Le résumé montre bien le principe du transfer learning. MobileNetV2 transforme notre image de 160 × 160 en 1280 feature maps de 5 × 5, et le GlobalAveragePooling réduit chacune à une seule valeur. Sur les 2 259 265 paramètres du modèle, 2 257 984 appartiennent à MobileNetV2 et sont gelés. Il n'en reste que **1 281 à entraîner** : les 1280 poids de la couche de sortie, plus son biais.

C'est plus de 1000 fois moins que les 1,4 million de paramètres entraînables de notre CNN. Avec si peu de paramètres à apprendre, le risque d'overfitting est très faible, même avec seulement 1600 images.

In [ ]:
tf.keras.utils.set_random_seed(42)
reset_training(mobilenet_model, mobilenet_initial_weights)

mobilenet_history = mobilenet_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=5,
    shuffle=False,
)

plot_history(mobilenet_history, "MobileNetV2")

#### Analyse des courbes

Le contraste avec nos CNN est frappant. Dès la première epoch, le modèle atteint 93,8 % en validation, alors que nos CNN démarraient autour de 50 %, c'est-à-dire au hasard. Il termine à **98 % en validation** après seulement 5 epochs, contre 69 à 76 % pour nos CNN après 15 epochs. La loss de validation passe de 0,27 à 0,08.

Le modèle n'a presque rien à apprendre : les convolutions de MobileNetV2 savent déjà extraire des caractéristiques très utiles (formes de tête, oreilles, museau, pelage), puisqu'elles ont été entraînées sur des millions d'images d'animaux. La seule chose qu'on entraîne, c'est la dernière couche, qui apprend à combiner ces 1280 caractéristiques pour distinguer un chat d'un chien.

On remarque que la validation reste au-dessus de l'entraînement pendant toutes les epochs (98 % contre 97 % à la fin). Ce n'est pas un problème, et ça s'explique de deux façons. D'abord, l'accuracy d'entraînement est une moyenne calculée pendant l'epoch, alors que le modèle s'améliore encore : c'est surtout visible à la première epoch (72,8 % en entraînement contre 93,8 % en validation), où le modèle part de zéro. Ensuite, le dropout est actif pendant l'entraînement et désactive 20 % des caractéristiques, ce qui rend la tâche un peu plus difficile qu'en validation, où toutes les caractéristiques sont utilisées.

Il n'y a donc **aucun overfitting** : le modèle réussit au moins aussi bien sur les images qu'il n'a jamais vues que sur celles d'entraînement. Les courbes continuent même de progresser légèrement à la 5e epoch, mais le gain devient très faible (de 97,9 % à 98 % entre les epochs 4 et 5), donc entraîner plus longtemps n'apporterait presque rien.

Seul point négatif : chaque epoch prend environ 6 à 9 secondes, contre 2 à 4 pour notre CNN de base. Même si on n'entraîne qu'une seule couche, chaque image doit passer entièrement à travers MobileNetV2, qui est un réseau beaucoup plus profond. Sur 5 epochs, l'entraînement dure quand même à peu près aussi longtemps que celui de notre baseline sur 15 epochs.

In [ ]:
mobilenet_results = evaluate_model(mobilenet_model, test_dataset, "MobileNetV2")

In [ ]:
comparison = pd.DataFrame(
    [baseline_results, augmented_results, regularized_results, mobilenet_results]
).set_index("model")
comparison["écart train/val (dernière epoch)"] = [
    training_history.history["accuracy"][-1] - training_history.history["val_accuracy"][-1]
    for training_history in [baseline_history, augmented_history, regularized_history, mobilenet_history]
]
display(comparison.style.format("{:.1%}"))

comparison[["accuracy", "f1"]].plot.bar(figsize=(11, 5), rot=0)
plt.title("Accuracy et F1-score sur le test set")
plt.ylabel("score")
plt.ylim(0, 1)
plt.show()

## Conclusion générale

| Modèle | Accuracy | F1-score | Écart train/val | Paramètres entraînables |
|---|---|---|---|---|
| Baseline | 77,2 % | 80,0 % | 11,8 pts | 1 424 801 |
| Data augmentation | 73,5 % | 78,2 % | 4,2 pts | 1 424 801 |
| Dropout + BatchNorm | 78,8 % | 79,7 % | 3,5 pts | 1 425 041 |
| **MobileNetV2** | **96,2 %** | **96,6 %** | **-1,0 pt** | **1 281** |

Pour MobileNetV2, l'écart train/val est négatif parce que l'accuracy de validation est au-dessus de celle d'entraînement : le modèle ne fait aucun overfitting.

Ce projet nous a permis de suivre toute la démarche de construction d'un classifieur d'images, et d'en tirer trois enseignements.

**Un CNN entraîné depuis zéro apprend, mais généralise mal avec peu de données.** Notre modèle de base reconnaît environ trois images sur quatre. Mais avec 1600 images pour 1,4 million de paramètres, il fait de l'overfitting : il apprend en partie les images d'entraînement par cœur, au lieu de règles générales. Le calcul des paramètres nous a montré que le problème venait surtout de la couche dense après le Flatten, qui en concentre 93 %.

**La régularisation réduit l'overfitting, mais ne remplace pas les données.** La data augmentation, le dropout et la batch normalization divisent par trois environ l'écart entre entraînement et validation. Mais sur le test set, les performances restent du même ordre que celles du baseline, entre 73 et 79 % en 15 epochs : ces techniques empêchent le modèle de mémoriser, sans lui apporter de nouvelles connaissances.

**Le transfer learning change tout.** En réutilisant un modèle déjà entraîné sur plus d'un million d'images, on passe à 96,2 % d'accuracy, en n'entraînant que 1281 paramètres pendant 5 epochs. C'est la méthode la plus adaptée quand on dispose de peu de données, et c'est celle qu'on utiliserait en pratique.

Au-delà des résultats, le projet nous a aussi appris à interpréter des courbes d'apprentissage, à lire une matrice de confusion au-delà de la seule accuracy, et à rester prudents face aux écarts de quelques points : avec un seul entraînement par modèle et 400 images de test, une partie des différences entre nos CNN peut venir du hasard. Le problème rencontré avec la batch normalization, bloquée à 50 % en validation avant qu'on corrige son momentum, nous a aussi montré qu'un bon score d'entraînement ne garantit rien tant qu'on n'a pas vérifié la validation.

## Comparaison avec un modèle plus récent : EfficientNetV2S

MobileNetV2 date de 2018, et il fait partie des modèles les moins précis sur ImageNet (71,3 % de top-1). On a voulu vérifier si un modèle plus récent et plus performant ferait mieux sur notre problème.

Pour choisir, Claude (l'assistant IA d'Anthropic que nous avons utilisé) nous a indiqué la page [Keras Applications](https://keras.io/api/applications/), qui compare tous les modèles pré-entraînés disponibles dans Keras : taille, nombre de paramètres, précision sur ImageNet et temps de calcul.

| Modèle | Top-1 ImageNet | Paramètres | Taille |
|---|---|---|---|
| ConvNeXtXLarge (le plus précis) | 86,7 % | 350,1 M | 1,3 Go |
| EfficientNetV2L | 85,7 % | 119,0 M | 479 Mo |
| **EfficientNetV2S** | **83,9 %** | **21,6 M** | **88 Mo** |
| MobileNetV2 | 71,3 % | 3,5 M | 14 Mo |

On n'a pas pris le modèle le plus précis du tableau (ConvNeXtXLarge), pour trois raisons :

- **Le temps de calcul.** On entraîne sur CPU, puisque TensorFlow ne prend plus en charge le GPU sous Windows en natif. D'après les mesures de Keras, un modèle comme EfficientNetB7 (67 millions de paramètres) est déjà environ 60 fois plus lent que MobileNetV2 sur CPU. ConvNeXtXLarge, avec 350 millions de paramètres, le serait encore bien plus : chaque epoch prendrait trop de temps.
- **Un gain très faible.** EfficientNetV2S n'est qu'à 2,8 points de ConvNeXtXLarge sur ImageNet, alors qu'il est 16 fois plus léger. Au-delà d'un certain point, chaque point de précision coûte énormément en calcul.
- **Une tâche plus simple qu'ImageNet.** Le top-1 ImageNet mesure la capacité à distinguer 1000 catégories, dont plus d'une centaine de races de chiens très proches. Pour seulement deux classes aussi différentes que "chat" et "chien", un modèle de taille moyenne suffit largement.

EfficientNetV2S est donc le meilleur compromis entre précision et coût de calcul pour notre situation.

On reprend exactement la même méthode qu'avec MobileNetV2 : modèle gelé, même tête de classification (GlobalAveragePooling, Dropout de 0,2, une sortie sigmoid), même learning rate et 5 epochs. Aucun des deux modèles n'est fine-tuné, pour que la comparaison soit équitable. Deux différences techniques :
- EfficientNetV2S intègre sa propre normalisation des pixels : on lui donne donc les images directement, sans couche `Rescaling`.
- Il accepte n'importe quelle taille d'image, donc on garde nos images en 150 × 150, sans redimensionnement.

In [ ]:
tf.keras.utils.set_random_seed(42)

efficientnet_base = tf.keras.applications.EfficientNetV2S(
    input_shape=(150, 150, 3),
    include_top=False,
    weights="imagenet",
)
efficientnet_base.trainable = False

inputs = layers.Input(shape=(150, 150, 3))
features = efficientnet_base(inputs, training=False)
features = layers.GlobalAveragePooling2D()(features)
features = layers.Dropout(0.2)(features)
outputs = layers.Dense(1, activation="sigmoid")(features)

efficientnet_model = models.Model(inputs, outputs)

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
efficientnet_initial_weights = efficientnet_model.get_weights()

efficientnet_model.summary()

In [ ]:
tf.keras.utils.set_random_seed(42)
reset_training(efficientnet_model, efficientnet_initial_weights)

efficientnet_history = efficientnet_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=5,
    shuffle=False,
)

plot_history(efficientnet_history, "EfficientNetV2S")

In [ ]:
efficientnet_results = evaluate_model(efficientnet_model, test_dataset, "EfficientNetV2S")

In [ ]:
comparison = pd.DataFrame(
    [baseline_results, augmented_results, regularized_results, mobilenet_results, efficientnet_results]
).set_index("model")
comparison["écart train/val (dernière epoch)"] = [
    training_history.history["accuracy"][-1] - training_history.history["val_accuracy"][-1]
    for training_history in
    [baseline_history, augmented_history, regularized_history, mobilenet_history, efficientnet_history]
]
display(comparison.style.format("{:.1%}"))

comparison[["accuracy", "f1"]].plot.bar(figsize=(13, 5), rot=0)
plt.title("Accuracy et F1-score sur le test set")
plt.ylabel("score")
plt.ylim(0, 1)
plt.show()

## Conclusion générale

| Modèle | Accuracy | F1-score | Écart train/val | Paramètres entraînables |
|---|---|---|---|---|
| Baseline | 77,2 % | 80,0 % | 11,8 pts | 1 424 801 |
| Data augmentation | 73,5 % | 78,2 % | 4,2 pts | 1 424 801 |
| Dropout + BatchNorm | 78,8 % | 79,7 % | 3,5 pts | 1 425 041 |
| MobileNetV2 | 96,2 % | 96,6 % | -1,0 pt | 1 281 |
| **EfficientNetV2S** | **98,8 %** | **98,9 %** | **-0,6 pt** | **1 281** |

L'accuracy et le F1-score sont mesurés sur le test set (400 images). Pour les deux modèles pré-entraînés, l'écart train/val est négatif parce que leur accuracy de validation est au-dessus de celle d'entraînement : ils ne font aucun overfitting.

Ce projet nous a permis de suivre toute la démarche de construction d'un classifieur d'images, et d'en tirer trois enseignements.

**Un CNN entraîné depuis zéro apprend, mais généralise mal avec peu de données.** Notre modèle de base reconnaît environ trois images sur quatre. Mais avec 1600 images pour 1,4 million de paramètres, il fait de l'overfitting : il apprend en partie les images d'entraînement par cœur, au lieu de règles générales. Le calcul des paramètres nous a montré que le problème venait surtout de la couche dense après le Flatten, qui en concentre 93 %.

**La régularisation réduit l'overfitting, mais ne remplace pas les données.** La data augmentation, le dropout et la batch normalization divisent par trois environ l'écart entre entraînement et validation. Mais sur le test set, les performances restent du même ordre que celles du baseline, entre 73 et 79 % en 15 epochs : ces techniques empêchent le modèle de mémoriser, sans lui apporter de nouvelles connaissances.

**Le transfer learning change tout.** En réutilisant un modèle déjà entraîné sur plus d'un million d'images, on passe de 78,8 % à 96,2 % avec MobileNetV2, puis à **98,8 % avec EfficientNetV2S**, qui ne se trompe que sur 5 images sur 400. Dans les deux cas, on n'entraîne que 1281 paramètres pendant 5 epochs. Le choix du modèle pré-entraîné compte aussi : EfficientNetV2S, plus récent et plus précis sur ImageNet, fait trois fois moins d'erreurs que MobileNetV2, au prix d'un entraînement environ 4 fois plus lent. Il nous a semblé être le meilleur compromis entre précision et coût de calcul sur notre CPU, plutôt que les modèles les plus précis du tableau Keras, beaucoup trop lourds.

Au-delà des résultats, le projet nous a appris à interpréter des courbes d'apprentissage, à lire une matrice de confusion au-delà de la seule accuracy, et à rester prudents face aux écarts de quelques points : avec un seul entraînement par modèle et 400 images de test, une partie des différences peut venir du hasard. On l'a d'ailleurs constaté avec EfficientNetV2S, à égalité avec MobileNetV2 en validation, mais nettement meilleur sur le test. Le problème rencontré avec la batch normalization, bloquée à 50 % en validation avant qu'on corrige son momentum, nous a aussi montré qu'un bon score d'entraînement ne garantit rien tant qu'on n'a pas vérifié la validation.